In [1]:
import os
import sys
import mlflow
import mlflow.tensorflow
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

# Import your custom logger and exception handler from the local files
from logger import logging
from exception import CustomException

logging.info("Imports completed successfully.")

In [2]:
# --- Define all constants and paths here ---

# UPDATED: Paths now match your project structure
DATA_DIR = 'Img_data/chest_xray'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Model parameters
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

# MLflow setup
MLFLOW_EXPERIMENT_NAME = "Chest_XRay_Pneumonia_Classification"

logging.info("Configuration loaded.")

In [4]:
import os

# --- Check Training Set ---
num_normal_train = len(os.listdir(os.path.join(TRAIN_DIR, 'NORMAL')))
num_pneumonia_train = len(os.listdir(os.path.join(TRAIN_DIR, 'PNEUMONIA')))
print("Training Set Image Counts:")
print(f"  - NORMAL: {num_normal_train}")
print(f"  - PNEUMONIA: {num_pneumonia_train}")

# --- Check Validation Set ---
num_normal_val = len(os.listdir(os.path.join(VAL_DIR, 'NORMAL')))
num_pneumonia_val = len(os.listdir(os.path.join(VAL_DIR, 'PNEUMONIA')))
print("\nValidation Set Image Counts:")
print(f"  - NORMAL: {num_normal_val}")
print(f"  - PNEUMONIA: {num_pneumonia_val}")

# --- Check Test Set ---
num_normal_test = len(os.listdir(os.path.join(TEST_DIR, 'NORMAL')))
num_pneumonia_test = len(os.listdir(os.path.join(TEST_DIR, 'PNEUMONIA')))
print("\nTest Set Image Counts:")
print(f"  - NORMAL: {num_normal_test}")
print(f"  - PNEUMONIA: {num_pneumonia_test}")

Training Set Image Counts:
  - NORMAL: 1341
  - PNEUMONIA: 3875

Validation Set Image Counts:
  - NORMAL: 8
  - PNEUMONIA: 8

Test Set Image Counts:
  - NORMAL: 234
  - PNEUMONIA: 390


In [5]:
try:
    logging.info("Setting up ImageDataGenerators...")
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    validation_test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

    train_generator = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary'
    )

    validation_generator = validation_test_datagen.flow_from_directory(
        VAL_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='binary'
    )
    logging.info(f"Data loaded from {DATA_DIR} successfully.")
except Exception as e:
    raise CustomException(e, sys)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.


In [6]:
try:
    logging.info("Building the CNN model architecture...")
    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.summary()
    logging.info("CNN model built successfully.")
except Exception as e:
    raise CustomException(e, sys)



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 74, 74, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 17, 17, 128)      

In [7]:
mlflow.set_experiment("Image classification")

with mlflow.start_run() as run:
    try:
        logging.info("Starting a new MLflow run for model training.")
        
        mlflow.log_param("epochs", EPOCHS)
        mlflow.log_param("learning_rate", LEARNING_RATE)
        mlflow.log_param("batch_size", BATCH_SIZE)
        
        model.compile(
            loss='binary_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
            metrics=['accuracy']
        )
        
        history = model.fit(
            train_generator,
            epochs=EPOCHS,
            validation_data=validation_generator,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)]
        )
        
        logging.info("Model training complete.")
        
        mlflow.log_metric("final_validation_accuracy", max(history.history['val_accuracy']))
        
        pd.DataFrame(history.history).plot(figsize=(10, 6))
        plt.grid(True)
        plt.title("Model Training History")
        history_plot_path = "training_history.png"
        plt.savefig(history_plot_path)
        mlflow.log_artifact(history_plot_path, "plots")
        plt.close()

        mlflow.tensorflow.log_model(model, "model")
        
        logging.info(f"Run completed and logged with ID: {run.info.run_id}")
        
    except Exception as e:
        logging.error("An error occurred during model training.")
        raise CustomException(e, sys)

run_id = run.info.run_id

2025/09/02 11:11:57 INFO mlflow.tracking.fluent: Experiment with name 'Image classification' does not exist. Creating a new experiment.


Epoch 1/20


163/163 [==============================] - 229s 1s/step - loss: 0.4535 - accuracy: 0.7897 - val_loss: 0.6642 - val_accuracy: 0.6250
Epoch 2/20
163/163 [==============================] - 163s 996ms/step - loss: 0.3309 - accuracy: 0.8520 - val_loss: 0.4819 - val_accuracy: 0.8750
Epoch 3/20
163/163 [==============================] - 156s 956ms/step - loss: 0.2759 - accuracy: 0.8773 - val_loss: 0.6016 - val_accuracy: 0.7500
Epoch 4/20
 29/163 [====>.........................] - ETA: 1:59 - loss: 0.2569 - accuracy: 0.8922

KeyboardInterrupt: 